# Phase 4-1: Detection Performance Analysis

Run inference on all datasets, collect stats, make histograms and sample images.

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

# colab setup
if 'google.colab' in sys.modules:
    !pip install ultralytics -q
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/Colab Notebooks/data"
    STAGING = "/content/staged_data"
else:
    BASE = "/Users/tyreecruse/Desktop/CS230/Project/Data"
    STAGING = BASE

import numpy as np
import pandas as pd
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
MODEL_PATH = f"{BASE}/training/results from training/weights/best.pt"
ZIPS_DIR = f"{BASE}/analysis/zips"
FIGURES_DIR = f"{BASE}/analysis/figures"
STATS_DIR = f"{FIGURES_DIR}/performance_stats"
HIST_DIR = f"{STATS_DIR}/histograms"
SAMPLES_DIR = f"{STATS_DIR}/samples"

os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(HIST_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
os.makedirs(STAGING, exist_ok=True)

datasets = [
    'clean',
    'fgsm_030', 'fgsm_045', 'fgsm_060', 'fgsm_075', 'fgsm_090', 'fgsm_105',
    'gaussian_010', 'gaussian_050', 'gaussian_150', 'gaussian_200', 'gaussian_250',
    'patches',
]

print(f"Datasets: {len(datasets)}")

In [ ]:
model = YOLO(MODEL_PATH)
print("Model loaded")

In [ ]:
# run inference on everything
allStats = []

for ds in datasets:
    print(f"\n{'='*50}")
    print(f"{ds}")
    print('='*50)
    
    # unzip if needed
    zipPath = f"{ZIPS_DIR}/{ds}.zip"
    stagedPath = f"{STAGING}/{ds}"
    if not os.path.exists(stagedPath):
        print("unzipping...")
        with zipfile.ZipFile(zipPath, 'r') as zf:
            zf.extractall(STAGING)
    
    # find images dir - try a bunch of places
    imagesDir = None
    for candidate in [
        f"{stagedPath}/images/test",
        f"{stagedPath}/test/images",
        f"{stagedPath}/images",
        f"{stagedPath}/{ds}/images",
        stagedPath,
    ]:
        if os.path.exists(candidate):
            files = list(Path(candidate).glob('*.jpg')) + list(Path(candidate).glob('*.png'))
            if files:
                imagesDir = candidate
                break
    
    if imagesDir is None:
        # search recursively
        for d in Path(stagedPath).rglob('images'):
            files = list(d.glob('*.jpg')) + list(d.glob('*.png'))
            if files:
                imagesDir = str(d)
                break
    
    if imagesDir is None:
        print("  no images found!")
        continue
    
    # find labels dir (same logic)
    labelsDir = None
    for candidate in [
        f"{stagedPath}/labels/test",
        f"{stagedPath}/test/labels",
        f"{stagedPath}/labels",
        f"{stagedPath}/{ds}/labels",
    ]:
        if os.path.exists(candidate) and list(Path(candidate).glob('*.txt')):
            labelsDir = candidate
            break
    
    # get all images
    imgFiles = list(Path(imagesDir).glob('*.jpg')) + list(Path(imagesDir).glob('*.png'))
    print(f"  {len(imgFiles)} images")
    
    # run inference on each
    dsStats = []
    for imgPath in tqdm(imgFiles, desc=f"  {ds}"):
        img = np.array(Image.open(imgPath).convert('RGB'))
        h, w = img.shape[:2]
        
        # inference
        results = model(img, conf=0.25, verbose=False)
        
        # extract detections
        dets = []
        for r in results:
            for i in range(len(r.boxes)):
                dets.append({
                    'bbox': r.boxes.xyxy[i].cpu().numpy().tolist(),
                    'conf': float(r.boxes.conf[i].cpu()),
                })
        
        # load ground truth if we have labels
        gts = []
        if labelsDir:
            lblPath = Path(labelsDir) / (imgPath.stem + '.txt')
            if lblPath.exists():
                with open(lblPath) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            xc = float(parts[1]) * w
                            yc = float(parts[2]) * h
                            bw = float(parts[3]) * w
                            bh = float(parts[4]) * h
                            gts.append([xc-bw/2, yc-bh/2, xc+bw/2, yc+bh/2])
        
        # stats
        maxConf = max([d['conf'] for d in dets]) if dets else 0
        nDets = len(dets)
        
        # best iou
        bestIou = 0
        if dets and gts:
            for d in dets:
                for g in gts:
                    b1, b2 = d['bbox'], g
                    x1 = max(b1[0], b2[0])
                    y1 = max(b1[1], b2[1])
                    x2 = min(b1[2], b2[2])
                    y2 = min(b1[3], b2[3])
                    inter = max(0, x2-x1) * max(0, y2-y1)
                    union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
                    iou = inter/union if union > 0 else 0
                    bestIou = max(bestIou, iou)
        
        dsStats.append({
            'dataset': ds,
            'filename': imgPath.name,
            'n_detections': nDets,
            'max_conf': maxConf,
            'best_iou': bestIou,
        })
    
    allStats.extend(dsStats)
    print(f"  done: {len(dsStats)} images")

print(f"\nTotal: {len(allStats)}")

In [ ]:
df = pd.DataFrame(allStats)
df.to_csv(f"{STATS_DIR}/inference_stats.csv", index=False)
print(f"saved inference_stats.csv ({len(df)} rows)")

In [ ]:
# make histograms for each dataset
print("Making histograms...")

for ds in datasets:
    subset = df[df['dataset'] == ds]
    if len(subset) == 0:
        continue
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # confidence
    axes[0].hist(subset['max_conf'], bins=50, color='blue', alpha=0.7)
    axes[0].axvline(subset['max_conf'].mean(), color='red', ls='--', 
                   label=f"mean={subset['max_conf'].mean():.3f}")
    axes[0].set_xlabel('Max Confidence')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'{ds}: Confidence')
    axes[0].legend()
    
    # iou
    axes[1].hist(subset['best_iou'], bins=50, color='green', alpha=0.7)
    axes[1].axvline(subset['best_iou'].mean(), color='red', ls='--',
                   label=f"mean={subset['best_iou'].mean():.3f}")
    axes[1].set_xlabel('Best IoU')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'{ds}: IoU')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(f"{HIST_DIR}/{ds}.png", dpi=150)
    plt.close()

print(f"saved to {HIST_DIR}")

In [ ]:
# save sample images with detections drawn
print("\nSaving sample images...")

for ds in datasets:
    subset = df[df['dataset'] == ds].copy()
    if len(subset) < 3:
        continue
    
    os.makedirs(f"{SAMPLES_DIR}/{ds}", exist_ok=True)
    
    # find images dir again (copy-paste from above)
    stagedPath = f"{STAGING}/{ds}"
    imagesDir = None
    for candidate in [
        f"{stagedPath}/images/test", f"{stagedPath}/test/images",
        f"{stagedPath}/images", f"{stagedPath}/{ds}/images", stagedPath
    ]:
        if os.path.exists(candidate):
            files = list(Path(candidate).glob('*.jpg')) + list(Path(candidate).glob('*.png'))
            if files:
                imagesDir = candidate
                break
    if imagesDir is None:
        for d in Path(stagedPath).rglob('images'):
            if list(d.glob('*.jpg')) or list(d.glob('*.png')):
                imagesDir = str(d)
                break
    
    if imagesDir is None:
        continue
    
    for metric in ['max_conf', 'best_iou']:
        sortedDf = subset.sort_values(metric)
        
        samples = {
            'low': sortedDf.iloc[0],
            'median': sortedDf.iloc[len(sortedDf)//2],
            'high': sortedDf.iloc[-1],
        }
        
        for level, row in samples.items():
            imgPath = Path(imagesDir) / row['filename']
            if not imgPath.exists():
                continue
            
            img = np.array(Image.open(imgPath).convert('RGB'))
            
            # run inference and draw boxes
            results = model(img, conf=0.25, verbose=False)
            for r in results:
                for i in range(len(r.boxes)):
                    x1, y1, x2, y2 = [int(c) for c in r.boxes.xyxy[i].cpu().numpy()]
                    conf = float(r.boxes.conf[i].cpu())
                    cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 3)
                    label = f"tank: {conf:.2f}"
                    cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            
            # save
            outPath = f"{SAMPLES_DIR}/{ds}/{metric}_{level}.png"
            Image.fromarray(img).save(outPath)
    
    print(f"  {ds}: done")

print("done")

In [ ]:
# summary table
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

summaryRows = []
for ds in datasets:
    subset = df[df['dataset'] == ds]
    if len(subset) == 0:
        continue
    
    # parse attack type
    if ds == 'clean':
        atype = 'Clean'
        strength = '-'
    elif 'fgsm' in ds:
        atype = 'FGSM'
        strength = f"{int(ds.split('_')[1])/1000:.3f}"
    elif 'gaussian' in ds:
        atype = 'Gaussian'
        strength = f"{int(ds.split('_')[1])/1000:.3f}"
    else:
        atype = 'Patch'
        strength = '-'
    
    summaryRows.append({
        'dataset': ds,
        'type': atype,
        'strength': strength,
        'n': len(subset),
        'mean_conf': subset['max_conf'].mean(),
        'median_conf': subset['max_conf'].median(),
        'mean_iou': subset['best_iou'].mean(),
        'det_rate': (subset['n_detections'] > 0).mean(),
    })

summaryDf = pd.DataFrame(summaryRows)
print(summaryDf.to_string(index=False))

summaryDf.to_csv(f"{STATS_DIR}/performance_summary.csv", index=False)
print(f"\nsaved performance_summary.csv")

In [ ]:
# list generated files
print(f"\nOutput: {STATS_DIR}")
print(f"Histograms: {len(list(Path(HIST_DIR).glob('*.png')))} files")
print("Samples:")
for d in sorted(Path(SAMPLES_DIR).iterdir()):
    if d.is_dir():
        print(f"  {d.name}: {len(list(d.glob('*.png')))} files")
print("CSVs:")
for f in Path(STATS_DIR).glob('*.csv'):
    print(f"  {f.name}: {f.stat().st_size/1024:.1f} KB")